In [0]:
%pip install --quiet --upgrade unitycatalog-ai
%restart_python

Creating Tables for the Agent for orders. The Agent will access the vector store for product inquiry. The agent will use the smartphones and inventory_stock tables for orders.

In [0]:
%run ../Includes/Lab_Setup_02

In [0]:
%sql
select * from smartphones;

In [0]:
%sql
select * from inventory_stock;

UDF for Getting Latest Phone

In [0]:
%sql

CREATE OR REPLACE FUNCTION get_latest_phone() RETURNS TABLE (phone_id INT, model STRING, color STRING, price INT)
COMMENT 'Returns the latest phone ordered by release date descending. Do not pass any arguments'
RETURN
  SELECT phone_id, model, color, price FROM smartphones ORDER BY release_date DESC LIMIT 1;

Create User Defined Function to check inventory

In [0]:
%sql

CREATE OR REPLACE FUNCTION check_inventory (phone_id INT COMMENT 'ID of the phone to check (integer)')
RETURNS TABLE (item_id INT, quantity INT)
COMMENT 'Checks the stock quantity for a given phone in the inventory'
RETURN SELECT item_id, quantity FROM inventory_stock WHERE item_id = phone_id;
    

Create a UDF that returns the identity of the current user interacting with the agent

In [0]:
%sql

CREATE OR REPLACE FUNCTION get_current_user() RETURNS STRING
COMMENT 'Returns the current session user. This is used for identifying the active user. IMPORTANT: This function accepts Zero parameters. Do not pass any arguments'
RETURN current_user();

Test the 3 functions created so far.

In [0]:
%sql

select * from workspace.genai_course.get_latest_phone();


In [0]:
%sql

select * from workspace.genai_course.check_inventory(4)

In [0]:
%sql

SELECT get_current_user();

Create a Python Function

In [0]:
def generate_checkout_url(user_id : str, phone_id : int, valid_quantity : int) -> str:
     
     from datetime import datetime, timedelta
     from urllib.parse import urlencode

     expiry_time = datetime.now() + timedelta(minutes = 30)
    
     params = {
         "u" : user_id,
         "i" : phone_id,
         "q" : valid_quantity,
         "t" : expiry_time
     }
     
     checkout_url = f"https://www.demomart.com/checkout?{urlencode(params)}"
     
     return checkout_url
     

Test the function

In [0]:
print(generate_checkout_url("ssagi@me.com", 1, 2));

We must create databricks function client to register this python function in Unity Catalog. We must provide the target catalog and schema

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()
catalog = "workspace"
schema = "genai_course"

generate_checkout_url.__doc__ = """Generates a checkout URL for the given user, phone, and quantity, valid for 30 minutes.

Args:
    user_id: The ID of the user.
    phone_id: The ID of the phone to purchase.
    valid_quantity: The quantity to purchase.
"""

python_tool_uc_info = client.create_python_function(
    func = generate_checkout_url,
    catalog = catalog,
    schema = schema,
    replace = True
)